In [1]:
import argparse
import json
import logging
import os
import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, Dataset
import h5py


_DIR  = Path("/workspace/VTCM_PYTHON/inverse_model")
_ROOT = Path(_DIR).parent
sys.path.insert(0, str(_DIR))

from inverse_config import InverseConfig
from inverse_architecture import InverseOperator
from inverse_trainer import InverseTrainer

In [2]:
class InverseH5Dataset(Dataset):
    """从 inverse_dataset_gen 导出的 HDF5 读取样本。"""

    def __init__(self, h5_path: str | Path):
        self.h5_path = Path(h5_path)
        if not self.h5_path.exists():
            raise FileNotFoundError(f"HDF5 file not found: {self.h5_path}")

        with h5py.File(self.h5_path, "r") as f:
            self.y = np.asarray(f["y"], dtype=np.float32)              # [N, T, n_s]
            self.u = np.asarray(f["u"], dtype=np.float32)              # [N, L, n_dir]
            self.c = np.asarray(f["c"], dtype=np.float32)              # [N, n_cond]
            self.x_query = np.asarray(f["x_query"], dtype=np.float32)  # [N, L]
            self.seq_lengths = (
                np.asarray(f["seq_lengths"], dtype=np.int64)
                if "seq_lengths" in f
                else None
            )

    def __len__(self) -> int:
        return self.y.shape[0]

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        sample = {
            "y": torch.from_numpy(self.y[idx]),
            "u": torch.from_numpy(self.u[idx]),
            "c": torch.from_numpy(self.c[idx]),
            "x_query": torch.from_numpy(self.x_query[idx]),
        }
        if self.seq_lengths is not None:
            sample["seq_lengths"] = torch.tensor(self.seq_lengths[idx], dtype=torch.long)
        return sample

In [3]:
def parse_args(argv=None) -> argparse.Namespace:
    p = argparse.ArgumentParser(description="PCNIO Training")
    p.add_argument("--epochs",       type=int,   default=100)
    p.add_argument("--batch_size",   type=int,   default=64)
    p.add_argument("--trunk_layers", type=int,   default=8,
                   help="TrunkDecoder 的 MLP 层数，>=1")
    p.add_argument("--physics_mode", type=str,   default="pinn",
                   choices=["none", "frf", "pinn", "both"])

    p.add_argument("--ckpt_dir",     type=str,   default=str(_DIR / "checkpoints"))
    p.add_argument("--resume",       action="store_true",
                   help="Resume training from ckpt_dir/best_model.pt")
    p.add_argument("--dataset_dir",  type=str,   default=str(_ROOT / "datasets" / "VTCM_inverse"),
                   help="inverse_dataset_gen 导出的数据集目录（包含 train.hdf5 / validation.hdf5 / test.hdf5）")
    p.add_argument("--use_full_seq", action="store_true",
                   help="训练和验证都读取 *_full_seq.hdf5（兼容旧用法）")
    p.add_argument("--train-full-seq", dest="train_full_seq", action=argparse.BooleanOptionalAction,
                   default=True, help="训练是否使用 *_full_seq.hdf5，默认关闭（训练采用窗口数据）")
    p.add_argument("--val-full-seq", dest="val_full_seq", action=argparse.BooleanOptionalAction,
                   default=True, help="验证是否使用 *_full_seq.hdf5，默认开启（验证采用完整序列）")
    p.add_argument("--log_every",    type=int,   default=5)
    p.add_argument("--val_every",    type=int,   default=5,
                   help="每val_every个epoch进行一次验证和可视化")
    return p.parse_args(argv)

In [4]:
physicsnemo_repo = "/workspace/VTCM_PYTHON/physicsnemo"
sys.path.insert(0, physicsnemo_repo)
sys.path = [str(p) for p in sys.path]

# 清理可能已加载的其他 physicsnemo 版本，避免命名冲突
for _m in list(sys.modules.keys()):
    if _m == "physicsnemo" or _m.startswith("physicsnemo."):
        del sys.modules[_m]

import hydra
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from physicsnemo.utils.logging import LaunchLogger
from physicsnemo.utils.checkpoint import save_checkpoint
from physicsnemo.models.fno import FNO
from omegaconf import DictConfig
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from pino_utils import VTCMHDF5MapStyleDataset

args = parse_args([])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
if torch.cuda.is_available():
        device = torch.device("cuda")
else:
        device = torch.device("cpu")
train_suffix = "_full_seq" if args.train_full_seq else ""
val_suffix = "_full_seq" if args.val_full_seq else ""
dataset_dir = Path(args.dataset_dir)
train_h5 = dataset_dir / f"train{train_suffix}.hdf5"
val_h5 = dataset_dir / f"validation{val_suffix}.hdf5"
train_h5

PosixPath('/workspace/VTCM_PYTHON/datasets/VTCM_inverse/train_full_seq.hdf5')

In [6]:
class InverseH5Dataset(Dataset):
    """从 inverse_dataset_gen 导出的 HDF5 读取样本。"""

    def __init__(self, h5_path: str | Path):
        self.h5_path = Path(h5_path)
        if not self.h5_path.exists():
            raise FileNotFoundError(f"HDF5 file not found: {self.h5_path}")

        with h5py.File(self.h5_path, "r") as f:
            self.y = np.asarray(f["y"], dtype=np.float32)              # [N, T, n_s]
            self.u = np.asarray(f["u"], dtype=np.float32)              # [N, L, n_dir]
            self.c = np.asarray(f["c"], dtype=np.float32)              # [N, n_cond]
            self.x_query = np.asarray(f["x_query"], dtype=np.float32)  # [N, L]
            self.seq_lengths = (
                np.asarray(f["seq_lengths"], dtype=np.int64)
                if "seq_lengths" in f
                else None
            )

    def __len__(self) -> int:
        return self.y.shape[0]

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        sample = {
            "y": torch.from_numpy(self.y[idx]),
            "u": torch.from_numpy(self.u[idx]),
            "c": torch.from_numpy(self.c[idx]),
            "x_query": torch.from_numpy(self.x_query[idx]),
        }
        if self.seq_lengths is not None:
            sample["seq_lengths"] = torch.tensor(self.seq_lengths[idx], dtype=torch.long)
        return sample

In [7]:
train_ds = InverseH5Dataset(train_h5)
val_ds = InverseH5Dataset(val_h5)

In [8]:
train_ds.y.shape, train_ds.u.shape, train_ds.c.shape, train_ds.x_query.shape

((160, 50000, 3), (160, 50000, 1), (160, 17), (160, 50000))

In [9]:
sample0 = train_ds[0]
n_sensors = int(sample0["y"].shape[-1])
spatial_len = int(sample0["x_query"].shape[0])
n_cond = int(sample0["c"].shape[-1])

In [10]:
train_loader = DataLoader(
        train_ds, batch_size=args.batch_size, shuffle=True,
        num_workers=0, pin_memory=(device.type == "cuda")
    )
val_loader = DataLoader(
        val_ds, batch_size=args.batch_size, shuffle=False,
        num_workers=0, pin_memory=(device.type == "cuda")
    ) if (val_ds is not None and len(val_ds) > 0) else None

In [11]:
in_channels = train_ds[0]["y"].shape[-1]
out_channels = train_ds[0]["u"].shape[-1]

In [12]:
model = FNO(
        in_channels=in_channels,
        out_channels=out_channels,
        decoder_layers=1,
        decoder_layer_size=64,
        dimension=1,
        latent_channels=64,
        num_fno_layers=6,
        num_fno_modes=12,
        padding=9,
    ).to(device)

optimizer = torch.optim.Adam(
        model.parameters(),
        betas=(0.9, 0.999),
        lr=0.0003,
        weight_decay=0.0,
    )

scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99948708)

In [13]:
def test_step(model, dataloader, epoch, device):
    model.eval()
    loss_epoch = 0.0
    n_batch= 0
    viz_pred = None
    viz_target = None

    with torch.no_grad():
        for batch in dataloader:
            input = batch["y"].to(device, non_blocking=True).transpose(1, 2)
            output = batch["u"].to(device, non_blocking=True).transpose(1, 2)
            pred = model(input)
            loss = F.mse_loss(pred, output)
            loss_epoch += loss.item()
            n_batch += 1
            if viz_pred is None:
                viz_pred = pred[0].detach().cpu().numpy()      # [C, T]
                viz_target = output[0].detach().cpu().numpy()  # [C, T]
    avg_loss = loss_epoch / max(n_batch, 1)
    if viz_pred is not None and (epoch % 10 == 0):
        os.makedirs("figures", exist_ok=True)
        plt.style.use("seaborn-v0_8-whitegrid")
        plt.rcParams.update(
            {
                "font.family": "serif",
                "font.size": 11,
                "axes.labelsize": 11,
                "axes.titlesize": 12,
                "legend.fontsize": 9,
                "lines.linewidth": 1.4,
                "figure.dpi": 150,
            }
        )
        # plot the pred values vs the truth, only vertical irre
        figure = plt.figure(figsize=(9, 7))
        component_offsets = [0, 3, 6]
        t_axis = np.arange(viz_pred.shape[1])
        plt.plot(t_axis, viz_target[0].T, label="Ground Truth", linestyle="--")
        plt.plot(t_axis, viz_pred[0].T, label="Prediction")
        plt.xlabel("Time Step")
        plt.ylabel("Value") 
        figure.tight_layout()
        figure.savefig(f"figures/val_epoch_{epoch:04d}.png", dpi=300)
        plt.close(figure)
    return avg_loss

        
           

In [14]:
epoches = 100
for epoch in range(epoches):
    total_loss = 0.0
    pbar = tqdm(
                train_loader,
                desc=f"Epoch {epoch + 1}/{epoches}",
                total=len(train_loader),
                leave=False,
            )
    for batch_idx, batch in enumerate(pbar):
        optimizer.zero_grad()
        input = batch["y"].to(device, non_blocking=True).transpose(1, 2)
        output = batch["u"].to(device, non_blocking=True).transpose(1, 2)
        pred = model(input)
        loss = F.mse_loss(pred, output)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4e}")
        scheduler.step()
    avg_loss = total_loss / len(train_loader)
    err = test_step(model, val_loader, epoch, device) 
    

In [21]:
# 推理模式，推理测试数据集的所有样本，并绘图
model.eval()
batch_num = 0 
with torch.no_grad():
    for batch in val_loader:
        input = batch["y"].to(device, non_blocking=True).transpose(1, 2)
        output = batch["u"].to(device, non_blocking=True).transpose(1, 2)
        pred = model(input)
        # 这里可以添加代码将 pred 和 output 保存或绘图
        os.makedirs("figures", exist_ok=True)
        plt.style.use("seaborn-v0_8-whitegrid")
        plt.rcParams.update(
            {
                "font.family": "serif",
                "font.size": 11,
                "axes.labelsize": 11,
                "axes.titlesize": 12,
                "legend.fontsize": 9,
                "lines.linewidth": 1.4,
                "figure.dpi": 150,
            }
        )
        for i in range(pred.shape[0]):
            # plot the pred values vs the truth, only vertical irre
            figure = plt.figure(figsize=(9, 7))
            t_axis = np.arange(output.shape[2])
            plt.plot(t_axis, output[i][0].cpu().T, label="Ground Truth", linestyle="--")
            plt.plot(t_axis, pred[i][0].cpu().T, label="Prediction")
            plt.xlabel("Time Step")
            plt.ylabel("Value") 
            figure.tight_layout()
            figure.savefig(f"figures/val_epoch_{epoch:04d}_batch_{batch_num:04d}.png", dpi=300)
            plt.close(figure)
            batch_num += 1